# mid_t Sensitivity Sweep — Lite

Single-checkpoint mid_t sweep for the COMP447 progress report.

- **Checkpoint**: 1980 kimg ECT (`network-snapshot-000198.pkl`)
- **mid_t values**: 8 points across [0.1, 2.5]
- **Samples per point**: 5000 (5k FID screening)
- **Expected runtime**: 15-25 minutes on Colab G4
- **Output**: `project/results/midt_sweep/lite_results.csv` + `lite_curve.png`

## Cell 1: Verify GPU and clone repo

In [ ]:
import os, subprocess, torch
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")

REPO = "/content/COMP447"
if not os.path.exists(REPO):
    print("Cloning repo...")
    subprocess.check_call(["git", "clone", "https://github.com/bakaraman/COMP447.git", REPO])
else:
    subprocess.check_call(["git", "-C", REPO, "fetch", "origin", "main"])
    subprocess.check_call(["git", "-C", REPO, "reset", "--hard", "origin/main"])
print("Repo synced to:", REPO)

## Cell 2a: Try to find the checkpoint (Drive or repo)

The 1980 kimg snapshot is 213 MB and gitignored — not in the cloned repo. This cell first tries the standard repo path (in case it was uploaded another way), then mounts Google Drive and globs for the file. If nothing is found, run **Cell 2b** instead and upload directly.

In [ ]:
import glob

CHECKPOINT = None

REPO_PATH = f"{REPO}/project/results_backup/ect_checkpoints/network-snapshot-000198.pkl"
if os.path.exists(REPO_PATH):
    CHECKPOINT = REPO_PATH

if CHECKPOINT is None:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        hits = glob.glob("/content/drive/MyDrive/**/network-snapshot-000198.pkl", recursive=True)
        if hits:
            CHECKPOINT = hits[0]
    except Exception as e:
        print(f"Drive mount skipped: {e}")

for extra in ["/content/network-snapshot-000198.pkl",
              "/content/checkpoint.pkl"]:
    if CHECKPOINT is None and os.path.exists(extra):
        CHECKPOINT = extra

if CHECKPOINT:
    print(f"Checkpoint found: {CHECKPOINT}")
    print(f"Size: {os.path.getsize(CHECKPOINT)/1e6:.1f} MB")
else:
    print("Checkpoint NOT found. Run Cell 2b to upload it directly from your local machine.")

## Cell 2b: Upload the checkpoint directly (fallback)

Only run this if Cell 2a did not find the checkpoint. Opens a file picker; choose `network-snapshot-000198.pkl` from `/Users/batuhankaraman/Downloads/COMP447/project/results_backup/ect_checkpoints/` on your laptop. Upload takes 1-3 minutes for 213 MB. After this completes, re-run Cell 2a to verify the path is now set.

In [ ]:
from google.colab import files
uploaded = files.upload()
for fname in uploaded.keys():
    src = f"/content/{fname}"
    if os.path.exists(src):
        CHECKPOINT = src
        print(f"Uploaded: {CHECKPOINT}  ({os.path.getsize(CHECKPOINT)/1e6:.1f} MB)")
        break

## Cell 3: Setup the ECT environment (one-time)

ECT brings pinned versions and a small monkey-patch for PyTorch 2.x. Idempotent — safe to re-run.

In [ ]:
setup_log = subprocess.run(
    ["bash", f"{REPO}/project/scripts/setup_ect.sh"],
    capture_output=True, text=True,
)
print(setup_log.stdout[-2000:])
if setup_log.returncode != 0:
    print("STDERR:", setup_log.stderr[-2000:])
    raise RuntimeError("setup_ect.sh failed")

## Cell 4: Run the lite sweep

8 mid_t values × 5000 samples × FID via EDM `fid.py`. Streams progress to the cell output. CSV is rewritten after each completed point so partial progress is recoverable if the kernel disconnects.

In [ ]:
import sys
assert CHECKPOINT and os.path.exists(CHECKPOINT), "CHECKPOINT not set — run Cell 2a or 2b first."
cmd = [
    "python3", f"{REPO}/project/scripts/midt_sweep_lite.py",
    "--checkpoint", CHECKPOINT,
    "--num", "5000",
    "--gen_batch", "64",
    "--fid_batch", "64",
    "--output_dir", "project/results/midt_sweep",
    "--repo_root", REPO,
]
print("$", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
proc.wait()
print("\nreturn code:", proc.returncode)

## Cell 5: Inspect the CSV

In [ ]:
import pandas as pd
csv_path = f"{REPO}/project/results/midt_sweep/lite_results.csv"
df = pd.read_csv(csv_path)
df_sorted = df.sort_values("fid").reset_index(drop=True)
print("All results (sorted by FID):")
print(df_sorted.to_string(index=False))
print("\nBest mid_t:", float(df_sorted.iloc[0]['mid_t']), "FID:", float(df_sorted.iloc[0]['fid']))
print("Default 0.821 FID:", float(df[df['mid_t']==0.821]['fid'].iloc[0]) if (df['mid_t']==0.821).any() else "--")

## Cell 6: Show the plot inline

In [ ]:
from IPython.display import Image, display
plot_path = f"{REPO}/project/results/midt_sweep/lite_curve.png"
display(Image(plot_path))

## Cell 7: Push results back to GitHub (optional)

After this finishes, run `git pull origin main` in your local terminal to bring the CSV and PNG back. Note: this only pushes the new `project/results/midt_sweep/` directory; the checkpoint pkl stays on Colab and is not pushed.

In [ ]:
rs = lambda cmd: subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(rs(["git", "add", "project/results/midt_sweep"]).stdout)
print(rs(["git", "-c", "user.email=colab@batuhan", "-c", "user.name=colab",
         "commit", "-m", "Add lite mid_t sweep results"]).stdout)
print("--- run locally: git pull origin main ---")